# Calidad y Limpieza de Datos

## Individual Household Electric Power Consumption

Este notebook tiene como objetivo analizar la calidad de los datos crudos obtenidos mediante el proceso reproducible de ingesta y, posteriormente, definir y aplicar decisiones de limpieza justificadas a partir de los hallazgos encontrados.

El análisis se dividirá en dos etapas:

1. **Diagnóstico de calidad de datos:** identificación de valores faltantes, duplicados, problemas temporales, tipos incorrectos, valores imposibles y otras situaciones que puedan afectar el modelado.
2. **Limpieza de datos:** aplicación de tratamientos reproducibles basados en los problemas identificados durante el diagnóstico.

Los datos originales almacenados en `data/raw/` no serán modificados. Cualquier transformación se realizará sobre una copia de trabajo y posteriormente será incorporada al pipeline reproducible del proyecto.

# Parte I - DATA QUALITY

## 1. Configuración del entorno y acceso a los datos

Se importan las librerías necesarias y se define la ruta al dataset crudo generado por el pipeline de ingesta. Se utilizan rutas relativas al proyecto para evitar dependencias de una computadora o usuario específico.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Dataset generado por el pipeline de ingesta
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "household_power_consumption.txt"

print(f"Dataset: {RAW_FILE}")
print(f"Existe: {RAW_FILE.exists()}")

Dataset: c:\Users\breid\Projects\Household-Power-Mlops\data\raw\household_power_consumption.txt
Existe: True


## 2. Inspección inicial del archivo crudo

Antes de cargar completamente el dataset, se inspecciona una pequeña muestra para conocer su estructura, nombres de columnas, formato y representación original de los valores. En esta etapa no se realiza ninguna transformación sobre los datos.

In [3]:
df_sample = pd.read_csv(
    RAW_FILE,
    sep=";",
    nrows=10
)

df_sample

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
5,16/12/2006,17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0
6,16/12/2006,17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0
7,16/12/2006,17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0
8,16/12/2006,17:32:00,3.668,0.510,233.99,15.8,0.0,1.0,17.0
9,16/12/2006,17:33:00,3.662,0.510,233.86,15.8,0.0,2.0,16.0


## 3. Carga del dataset para diagnóstico

Una vez verificada la estructura del archivo, se carga el conjunto de datos completo para realizar el diagnóstico de calidad. En esta etapa los datos se mantienen en su representación original. Esto permite identificar los símbolos utilizados para valores faltantes y detectar posibles problemas antes de convertir tipos de datos o aplicar transformaciones. La limpieza se realizará únicamente después de documentar los problemas encontrados.

In [4]:
df_raw = pd.read_csv(
    RAW_FILE,
    sep=";",
    dtype=str
)

print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")

Filas: 2,075,259
Columnas: 9


## 4. Inspección general de la estructura

Antes de analizar problemas específicos de calidad, se realiza una revisión general del dataset completo. El objetivo es verificar las dimensiones, nombres de las variables y observar algunos registros del inicio y del final del archivo. Esta revisión permite confirmar que la carga se realizó correctamente y conocer la estructura que será evaluada en las siguientes etapas.

In [5]:
print("Dimensiones del dataset:")
print(df_raw.shape)

print("\nColumnas:")
print(df_raw.columns.tolist())

display(df_raw.head())
display(df_raw.tail())

Dimensiones del dataset:
(2075259, 9)

Columnas:
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.000
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.000
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.000
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.000
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.000


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2075254,26/11/2010,20:58:00,0.946,0.000,240.430,4.000,0.000,0.000,0.000
2075255,26/11/2010,20:59:00,0.944,0.000,240.000,4.000,0.000,0.000,0.000
2075256,26/11/2010,21:00:00,0.938,0.000,239.820,3.800,0.000,0.000,0.000
2075257,26/11/2010,21:01:00,0.934,0.000,239.700,3.800,0.000,0.000,0.000
2075258,26/11/2010,21:02:00,0.932,0.000,239.550,3.800,0.000,0.000,0.000


## 5. Diagnóstico de valores faltantes

Antes de realizar cualquier tratamiento sobre los datos, se identifican las distintas formas en que pueden estar representados los valores faltantes en el archivo original. Debido a que el dataset fue cargado inicialmente como texto para preservar su representación original, se revisarán tanto los valores nulos reconocidos automáticamente por pandas como posibles símbolos utilizados para representar datos ausentes. En esta etapa únicamente se realiza el diagnóstico. Las decisiones de limpieza se tomarán posteriormente considerando también la estructura temporal de los datos.

In [6]:
# Valores nulos reconocidos automáticamente por pandas
null_counts = df_raw.isna().sum()

print("Valores nulos detectados por pandas:")
print(null_counts)

Valores nulos detectados por pandas:
Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64


In [7]:
# Columnas que deberían contener mediciones numéricas
numeric_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

# Identificar valores que no pueden convertirse a número
non_numeric_values = {}

for column in numeric_columns:
    converted = pd.to_numeric(df_raw[column], errors="coerce")

    # Valores que originalmente existen, pero no pudieron convertirse
    mask = converted.isna() & df_raw[column].notna()

    values = df_raw.loc[mask, column].value_counts()

    if not values.empty:
        non_numeric_values[column] = values

non_numeric_values

{'Global_active_power': Global_active_power
 ?    25979
 Name: count, dtype: int64,
 'Global_reactive_power': Global_reactive_power
 ?    25979
 Name: count, dtype: int64,
 'Voltage': Voltage
 ?    25979
 Name: count, dtype: int64,
 'Global_intensity': Global_intensity
 ?    25979
 Name: count, dtype: int64,
 'Sub_metering_1': Sub_metering_1
 ?    25979
 Name: count, dtype: int64,
 'Sub_metering_2': Sub_metering_2
 ?    25979
 Name: count, dtype: int64}

In [8]:
# Filas con valores no numéricos en las seis variables afectadas
question_mask = df_raw[numeric_columns].eq("?").any(axis=1)

# Filas con valores nulos en Sub_metering_3
sub3_null_mask = df_raw["Sub_metering_3"].isna()

print(f"Filas con '?': {question_mask.sum():,}")
print(f"Filas con NaN en Sub_metering_3: {sub3_null_mask.sum():,}")

print(
    "¿Ocurren exactamente en las mismas filas?:",
    question_mask.equals(sub3_null_mask)
)

Filas con '?': 25,979
Filas con NaN en Sub_metering_3: 25,979
¿Ocurren exactamente en las mismas filas?: True


In [9]:
# Revisión de posibles valores faltantes en Date y Time
for column in ["Date", "Time"]:
    values = df_raw[column].astype("string").str.strip()

    null_count = df_raw[column].isna().sum()
    empty_count = values.eq("").sum()
    question_count = values.eq("?").sum()

    print(f"\nColumna: {column}")
    print(f"Valores NaN: {null_count:,}")
    print(f"Campos vacíos: {empty_count:,}")
    print(f"Símbolos '?': {question_count:,}")


Columna: Date
Valores NaN: 0
Campos vacíos: 0
Símbolos '?': 0

Columna: Time
Valores NaN: 0
Campos vacíos: 0
Símbolos '?': 0


### Interpretación de los valores faltantes

El diagnóstico permitió identificar 25,979 registros con valores faltantes, equivalentes aproximadamente al 1.25% de las 2,075,259 observaciones del dataset. La revisión no se limitó a buscar un símbolo específico. Primero se identificaron los valores nulos reconocidos automáticamente por pandas y posteriormente se evaluaron las variables que deberían contener mediciones numéricas para detectar valores que no pudieran convertirse a números. Se encontró que `Global_active_power`, `Global_reactive_power`, `Voltage`, `Global_intensity`, `Sub_metering_1` y `Sub_metering_2` contienen 25,979 apariciones del símbolo `?`, identificado como la representación de datos ausentes en estas variables. En `Sub_metering_3`, pandas reconoció automáticamente 25,979 campos vacíos como valores nulos (`NaN`). Además, se comprobó que ambas representaciones corresponden exactamente a las mismas 25,979 filas. Esto significa que en dichos registros están ausentes simultáneamente las siete variables de medición eléctrica. Por otra parte, `Date` y `Time` no presentan valores `NaN`, campos vacíos ni el símbolo `?`, por lo que no se identificaron valores faltantes en estas variables durante esta etapa. La validez de sus formatos y la correcta construcción de los timestamps se evaluarán posteriormente. Debido a que el dataset corresponde a una serie temporal, todavía no se eliminarán ni imputarán los registros afectados. Antes de definir una estrategia de limpieza será necesario analizar su distribución temporal, la duración de posibles períodos consecutivos sin mediciones y la frecuencia esperada de las observaciones.

## 6. Diagnóstico de registros duplicados

Los registros duplicados pueden introducir redundancia en el dataset, alterar los análisis posteriores y afectar el entrenamiento y la evaluación de los modelos. Debido a que los datos corresponden a una serie temporal, el diagnóstico se realizará desde dos perspectivas:

1. **Duplicados exactos:** registros en los que todas las columnas contienen exactamente los mismos valores.
2. **Timestamps duplicados:** observaciones que comparten la misma combinación de `Date` y `Time`, aunque los valores de las mediciones eléctricas sean diferentes.

La segunda comprobación es especialmente importante porque cada combinación de fecha y hora debería representar una única observación temporal. En esta etapa únicamente se identificarán y cuantificarán los posibles duplicados. No se eliminará ningún registro hasta analizar los resultados.

In [10]:
# Identificación de registros completamente duplicados
exact_duplicates = df_raw.duplicated()

print(f"Registros duplicados exactos: {exact_duplicates.sum():,}")
print(
    f"Porcentaje de duplicados exactos: "
    f"{exact_duplicates.mean() * 100:.4f}%"
)

Registros duplicados exactos: 0
Porcentaje de duplicados exactos: 0.0000%


In [11]:
# Identificación de timestamps duplicados
timestamp_duplicates = df_raw.duplicated(
    subset=["Date", "Time"],
    keep=False
)

print(
    f"Registros involucrados en timestamps duplicados: "
    f"{timestamp_duplicates.sum():,}"
)

if timestamp_duplicates.any():
    display(
        df_raw.loc[timestamp_duplicates]
        .sort_values(["Date", "Time"])
        .head(20)
    )

Registros involucrados en timestamps duplicados: 0


### Interpretación de los registros duplicados

No se identificaron registros completamente duplicados en las 2,075,259 observaciones del dataset. Esto indica que no existen filas repetidas que contengan exactamente los mismos valores en todas sus variables.

Debido a que el dataset corresponde a una serie temporal, también se verificó de manera independiente la existencia de timestamps duplicados utilizando la combinación de `Date` y `Time`. Esta comprobación tampoco identificó registros repetidos, por lo que cada combinación de fecha y hora aparece una única vez en el archivo.

Estos resultados indican que no es necesario aplicar ningún procedimiento de eliminación de duplicados durante la etapa de limpieza.

Sin embargo, la ausencia de timestamps duplicados no garantiza por sí sola que la secuencia temporal sea completamente regular. La continuidad de las observaciones y la existencia de posibles intervalos faltantes se evaluarán posteriormente mediante el análisis de gaps temporales y frecuencia.

## 7. Diagnóstico de fechas y tipos de datos

La correcta representación de las fechas y de las variables numéricas es fundamental para las etapas posteriores del proyecto, especialmente porque el problema será tratado como una serie temporal.

El dataset fue cargado inicialmente como texto para preservar su representación original durante el diagnóstico de calidad. Por esta razón, en esta sección se comprobará si la combinación de `Date` y `Time` puede convertirse correctamente a un timestamp y si las variables de medición eléctrica contienen valores compatibles con tipos numéricos.

Los valores faltantes identificados previamente (`?` y valores nulos) serán considerados como ausencias conocidas y no como errores de tipo.

En esta etapa las conversiones se utilizarán únicamente con fines de validación y no modificarán todavía el dataset original de trabajo.

In [12]:
# Construcción temporal del timestamp para validar Date y Time
datetime_test = pd.to_datetime(
    df_raw["Date"] + " " + df_raw["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

invalid_datetime = datetime_test.isna()

print(f"Timestamps evaluados: {len(datetime_test):,}")
print(f"Timestamps inválidos: {invalid_datetime.sum():,}")
print(
    f"Porcentaje de timestamps inválidos: "
    f"{invalid_datetime.mean() * 100:.4f}%"
)

if invalid_datetime.any():
    display(
        df_raw.loc[invalid_datetime, ["Date", "Time"]].head(20)
    )

Timestamps evaluados: 2,075,259
Timestamps inválidos: 0
Porcentaje de timestamps inválidos: 0.0000%


In [13]:
# Validación de las variables que deberían ser numéricas
numeric_type_validation = {}

for column in numeric_columns:
    # Excluir los valores faltantes ya identificados
    valid_values = df_raw[column].dropna()
    valid_values = valid_values[valid_values != "?"]

    # Intentar convertir el resto de los valores a tipo numérico
    converted = pd.to_numeric(valid_values, errors="coerce")

    # Contar valores que todavía no pudieron convertirse
    invalid_count = converted.isna().sum()

    numeric_type_validation[column] = invalid_count

numeric_type_validation = pd.Series(
    numeric_type_validation,
    name="Valores no convertibles"
)

numeric_type_validation

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
Name: Valores no convertibles, dtype: int64

### Interpretación de fechas y tipos de datos

La validación de la combinación de `Date` y `Time` se realizó sobre las 2,075,259 observaciones del dataset utilizando el formato esperado `día/mes/año hora:minuto:segundo`. No se identificaron timestamps inválidos, por lo que todas las observaciones presentan una representación temporal válida. Posteriormente, se evaluaron las siete variables de medición eléctrica. Para esta comprobación se excluyeron únicamente los valores faltantes previamente identificados (`?` y valores nulos), evitando clasificarlos incorrectamente como errores de tipo. Después de excluir estas ausencias conocidas, todos los valores restantes pudieron convertirse correctamente a valores numéricos. Por lo tanto, no se identificaron valores adicionales con formatos incompatibles en las variables eléctricas. Aunque el dataset fue cargado inicialmente como texto para preservar su representación original durante el diagnóstico, los resultados indican que `Date` y `Time` podrán transformarse posteriormente a una variable temporal y que las siete variables eléctricas podrán convertirse a tipos numéricos una vez definido el tratamiento de los valores faltantes. En consecuencia, no se requiere una corrección específica por fechas inválidas ni por valores numéricos mal formateados. Las conversiones definitivas de tipos se realizarán durante la etapa de limpieza.

## 8. Diagnóstico de continuidad y frecuencia temporal

En una serie temporal no es suficiente comprobar que los timestamps sean válidos y únicos. También es necesario verificar que las observaciones mantengan una frecuencia temporal consistente y detectar posibles intervalos sin registros. Una frecuencia irregular o la existencia de gaps puede afectar la construcción de variables temporales, la definición del horizonte de predicción y el entrenamiento posterior de los modelos. En esta sección se analizará la diferencia temporal entre observaciones consecutivas para determinar la frecuencia predominante del dataset e identificar posibles interrupciones en la secuencia temporal.Este análisis también permitirá complementar el diagnóstico de los valores faltantes, diferenciando entre timestamps que existen pero no contienen mediciones y períodos en los que directamente no existe un registro.

In [14]:
# Diferencia temporal entre observaciones consecutivas
time_differences = datetime_test.diff()

print("Diferencias temporales más frecuentes:")
print(
    time_differences
    .value_counts()
    .head(10)
)

Diferencias temporales más frecuentes:
0 days 00:01:00    2075258
Name: count, dtype: int64


In [16]:
# Convertir las diferencias temporales a segundos
time_differences_seconds = time_differences.dt.total_seconds()

# Un minuto equivale a 60 segundos
expected_frequency_seconds = 60

# Identificar intervalos diferentes a la frecuencia esperada
irregular_gaps = time_differences_seconds[
    time_differences_seconds.notna()
    & (time_differences_seconds != expected_frequency_seconds)
]

print(f"Intervalos temporales irregulares: {len(irregular_gaps):,}")

if not irregular_gaps.empty:
    print("\nDuración de los intervalos irregulares (segundos):")
    print(irregular_gaps.value_counts().sort_index().head(20))

Intervalos temporales irregulares: 0


### Interpretación de la continuidad y frecuencia temporal

El análisis de las diferencias entre timestamps consecutivos mostró que las 2,075,258 diferencias temporales existentes entre las 2,075,259 observaciones corresponden exactamente a intervalos de un minuto. A partir de este patrón se estableció una frecuencia temporal observada de 1 minuto y se verificó explícitamente la existencia de intervalos diferentes a dicha frecuencia. No se identificaron intervalos temporales irregulares.

Por lo tanto, la secuencia de timestamps es continua y mantiene una frecuencia regular de un minuto durante todo el período analizado. No se identificaron gaps producidos por la ausencia completa de timestamps. Este resultado permite distinguir los gaps temporales de los valores faltantes detectados previamente. Las 25,979 observaciones con mediciones eléctricas ausentes sí conservan su correspondiente `Date` y `Time`; por lo tanto, representan registros existentes con valores faltantes y no minutos ausentes de la serie temporal.

Esta distinción será considerada posteriormente durante la limpieza, ya que el tratamiento de mediciones faltantes dentro de una secuencia temporal continua requiere una estrategia diferente a la reconstrucción de timestamps inexistentes.

## 9. Diagnóstico de valores imposibles y outliers

La presencia de valores extremos no implica necesariamente que exista un problema de calidad. En variables de consumo eléctrico pueden presentarse observaciones poco frecuentes que correspondan a comportamientos reales y que, por lo tanto, no deberían eliminarse automáticamente. Por esta razón, el análisis se realizará en dos etapas:

1. **Valores potencialmente imposibles:** se buscarán observaciones incompatibles con la naturaleza de las variables, comenzando por valores negativos en las mediciones eléctricas.
2. **Outliers:** posteriormente se analizarán valores estadísticamente extremos para determinar su magnitud y distribución, sin asumir que representan errores.

Para realizar estas comprobaciones será necesario trabajar temporalmente con una representación numérica de las variables. Esta conversión se utilizará únicamente para el diagnóstico y no modificará `df_raw`.

In [17]:
# Copia temporal de las variables eléctricas para diagnóstico
df_numeric = df_raw[numeric_columns].copy()

# Los '?' identificados previamente se consideran valores faltantes
df_numeric = df_numeric.replace("?", np.nan)

# Conversión temporal a valores numéricos
df_numeric = df_numeric.apply(pd.to_numeric)

df_numeric.dtypes

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

In [18]:
negative_counts = (df_numeric < 0).sum()

print("Valores negativos por variable:")
print(negative_counts)

print(f"\nTotal de valores negativos: {negative_counts.sum():,}")

Valores negativos por variable:
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

Total de valores negativos: 0


In [19]:
# Resumen de los rangos observados en las variables eléctricas
ranges_summary = pd.DataFrame({
    "Mínimo": df_numeric.min(),
    "Máximo": df_numeric.max()
})

ranges_summary

,Mínimo,Máximo
Global_active_power,0.076,11.122
Global_reactive_power,0.000,1.390
Voltage,223.200,254.150
Global_intensity,0.200,48.400
Sub_metering_1,0.000,88.000
Sub_metering_2,0.000,80.000
Sub_metering_3,0.000,31.000


### Análisis de outliers

Una vez revisados los rangos observados y descartada la presencia de valores negativos, se analizarán posibles valores atípicos mediante el rango intercuartílico (IQR).
Este método permite identificar observaciones alejadas de la zona central de la distribución utilizando los cuartiles Q1 y Q3. Los valores detectados mediante este criterio serán considerados candidatos a outliers estadísticos y no errores automáticamente, ya que valores elevados de consumo eléctrico pueden representar comportamientos reales.

Por lo tanto, la identificación de outliers se utilizará como herramienta de diagnóstico antes de decidir si requieren algún tratamiento.

In [20]:
# Identificación de outliers mediante el método IQR
outlier_summary = {}

for column in numeric_columns:
    series = df_numeric[column].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outlier_mask = (series < lower_limit) | (series > upper_limit)

    outlier_summary[column] = {
        "Q1": q1,
        "Q3": q3,
        "Límite inferior": lower_limit,
        "Límite superior": upper_limit,
        "Cantidad outliers": outlier_mask.sum(),
        "Porcentaje outliers": outlier_mask.mean() * 100
    }

outlier_summary = pd.DataFrame(outlier_summary).T

outlier_summary

,Q1,Q3,Límite inferior,Límite superior,Cantidad outliers,Porcentaje outliers
Global_active_power,0.308,1.528,-1.522,3.358,94907.0,4.631236
Global_reactive_power,0.048,0.194,-0.171,0.413,40420.0,1.972400
Voltage,238.990,242.890,233.140,248.740,51067.0,2.491948
Global_intensity,1.400,6.400,-6.100,13.900,100961.0,4.926657
Sub_metering_1,0.000,0.000,0.000,0.000,169105.0,8.251923
Sub_metering_2,0.000,1.000,-1.500,2.500,77151.0,3.764786
Sub_metering_3,0.000,17.000,-25.500,42.500,0.0,0.000000


### Significado y unidades de las variables eléctricas

De acuerdo con la documentación oficial del dataset *Individual Household Electric Power Consumption* del UCI Machine Learning Repository, las variables eléctricas representan las siguientes mediciones:

| Variable | Unidad | Descripción |
|---|---|---|
| `Global_active_power` | kW | Potencia activa global del hogar promediada por minuto. |
| `Global_reactive_power` | kW | Potencia reactiva global del hogar promediada por minuto. |
| `Voltage` | V | Voltaje promediado por minuto. |
| `Global_intensity` | A | Intensidad de corriente global promediada por minuto. |
| `Sub_metering_1` | Wh | Energía activa correspondiente principalmente a la cocina. |
| `Sub_metering_2` | Wh | Energía activa correspondiente principalmente al cuarto de lavandería. |
| `Sub_metering_3` | Wh | Energía activa correspondiente principalmente al calentador eléctrico de agua y al aire acondicionado. |

Conocer el significado y las unidades de las variables es necesario antes de interpretar los valores extremos. Un valor identificado estadísticamente como outlier no se considerará automáticamente un error de calidad, ya que puede representar un pico real en el consumo eléctrico.

### Inspección de valores extremos

El método IQR identificó observaciones estadísticamente atípicas en varias variables. Sin embargo, un outlier estadístico no implica necesariamente un error de medición.

La documentación oficial proporciona el significado y las unidades de las variables, pero no establece límites de validez que permitan clasificar automáticamente los valores extremos como imposibles. Por esta razón, antes de tomar decisiones de limpieza se inspeccionará el comportamiento de la zona superior de cada distribución.

El objetivo es determinar si los máximos observados corresponden a valores completamente aislados o forman parte de un conjunto de mediciones elevadas presentes en los datos.

In [21]:
# Percentiles superiores para analizar el comportamiento de los valores extremos
extreme_percentiles = df_numeric.quantile(
    [0.95, 0.99, 0.995, 0.999, 1.0]
).T

extreme_percentiles.columns = [
    "P95",
    "P99",
    "P99.5",
    "P99.9",
    "Máximo"
]

extreme_percentiles

,P95,P99,P99.5,P99.9,Máximo
Global_active_power,3.264,4.850,5.518,6.790,11.122
Global_reactive_power,0.338,0.478,0.542,0.698,1.390
Voltage,245.940,248.270,249.070,250.630,254.150
Global_intensity,13.800,20.600,23.600,29.000,48.400
Sub_metering_1,1.000,38.000,39.000,44.000,88.000
Sub_metering_2,2.000,36.000,38.000,71.000,80.000
Sub_metering_3,19.000,20.000,29.000,30.000,31.000


### Inspección de los máximos observados

Los percentiles superiores muestran que algunas variables presentan máximos más alejados del resto de las observaciones de alta magnitud. Esta diferencia no permite clasificarlos directamente como errores, pero justifica inspeccionar los registros donde ocurren. Se revisarán los registros asociados a los máximos de cada variable para detectar posibles inconsistencias entre las mediciones antes de tomar una decisión sobre su tratamiento.

In [22]:
# Registros donde ocurre el máximo de cada variable
max_records = []

for column in numeric_columns:
    max_idx = df_numeric[column].idxmax()

    record = df_raw.loc[
        max_idx,
        ["Date", "Time"] + numeric_columns
    ].copy()

    record["Variable_máxima"] = column
    max_records.append(record)

max_records = pd.DataFrame(max_records)

max_records[
    ["Variable_máxima", "Date", "Time"] + numeric_columns
]

,Variable_máxima,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1150545,Global_active_power,22/2/2009,17:09:00,11.122,0.174,229.780,48.400,35.000,69.000,16.000
697040,Global_reactive_power,13/4/2008,18:44:00,6.342,1.390,231.770,28.000,36.000,7.000,17.000
1583871,Voltage,20/12/2009,15:15:00,0.300,0.000,254.150,1.200,0.000,0.000,0.000
1150545,Global_intensity,22/2/2009,17:09:00,11.122,0.174,229.780,48.400,35.000,69.000,16.000
1674525,Sub_metering_1,21/2/2010,14:09:00,7.430,0.142,234.330,32.000,88.000,0.000,17.000
1654413,Sub_metering_2,7/2/2010,14:57:00,5.170,0.090,246.350,21.000,0.000,80.000,1.000
730434,Sub_metering_3,6/5/2008,23:18:00,2.076,0.112,245.350,8.400,0.000,0.000,31.000


### Interpretación de valores extremos y outliers

El análisis mediante IQR identificó observaciones estadísticamente atípicas en varias de las variables eléctricas. Sin embargo, estos resultados no se interpretaron automáticamente como errores, debido a que un valor extremo puede corresponder a un pico real de consumo. La revisión de los percentiles superiores mostró que los valores máximos se encuentran por encima del comportamiento habitual de algunas variables, por lo que se inspeccionaron directamente los registros donde ocurren. No se observaron inconsistencias evidentes entre las mediciones. Por ejemplo, el máximo de `Global_active_power` (11.122 kW) y el máximo de `Global_intensity` (48.4 A) ocurren simultáneamente en el mismo registro, acompañados además por consumo en los distintos submedidores.

También se observó una limitación del método IQR en `Sub_metering_1`, donde Q1 y Q3 son iguales a cero debido a la alta concentración de observaciones con valor cero. Como consecuencia, cualquier valor superior a cero es clasificado estadísticamente como atípico, por lo que el criterio IQR no resulta adecuado por sí solo para determinar valores incorrectos en esta variable. Por estas razones, los outliers identificados se conservarán en esta etapa. No existe evidencia suficiente para considerarlos errores de calidad y su eliminación podría descartar picos reales de consumo que contienen información relevante para el comportamiento temporal de la serie.

## 10. Diagnóstico de distribución, correlaciones y riesgo de leakage

Como parte final del diagnóstico de calidad, se analizarán algunas características estadísticas y relaciones entre las variables que pueden afectar las etapas posteriores del proyecto. En primer lugar, se evaluará la asimetría (skewness) de las variables eléctricas. Esta medida permite identificar si una distribución es aproximadamente simétrica o si presenta una mayor concentración de observaciones hacia uno de sus extremos. Este análisis complementa la revisión de outliers realizada anteriormente y permite comprender mejor la estructura de las variables antes de definir transformaciones o preparar los datos para el modelado.

In [23]:
# Cálculo de la asimetría de las variables eléctricas
skewness = df_numeric.skew()

skewness_summary = pd.DataFrame({
    "Skewness": skewness
})

skewness_summary

,Skewness
Global_active_power,1.786233
Global_reactive_power,1.261914
Voltage,-0.326665
Global_intensity,1.849100
Sub_metering_1,5.944541
Sub_metering_2,7.090553
Sub_metering_3,0.724688


### Interpretación de la asimetría

Los resultados muestran que las variables eléctricas no presentan el mismo comportamiento distributivo. `Global_active_power`, `Global_reactive_power` y `Global_intensity` presentan asimetría positiva, indicando una mayor concentración de observaciones en valores relativamente bajos y una cola hacia mediciones más elevadas.

La mayor asimetría se observa en `Sub_metering_1` y `Sub_metering_2`, con valores de skewness de aproximadamente 5.95 y 7.09, respectivamente. Esto indica distribuciones fuertemente sesgadas hacia la derecha y es consistente con la presencia frecuente de consumos bajos o iguales a cero combinados con episodios de consumo considerablemente mayores.`Sub_metering_3` presenta una asimetría positiva menos pronunciada, mientras que `Voltage`, con un skewness cercano a cero (-0.33), muestra una distribución comparativamente más simétrica.

Estos resultados complementan el análisis de outliers anterior y muestran que la presencia de valores extremos está relacionada, al menos en parte, con la propia forma de las distribuciones. Por esta razón, la asimetría se documenta como una característica de los datos y no se aplicarán transformaciones automáticamente durante esta etapa de diagnóstico.

### Análisis de correlaciones

Se analizará la correlación entre las variables eléctricas para identificar relaciones lineales fuertes, posibles redundancias y dependencias que deban considerarse posteriormente durante la selección y construcción de características. La presencia de una correlación elevada no implica por sí sola que una variable deba eliminarse. Su interpretación dependerá del significado físico de las variables y del objetivo de modelado.

In [24]:
# Matriz de correlación de Pearson
correlation_matrix = df_numeric.corr()

correlation_matrix.round(3)

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Global_active_power,1.000,0.247,-0.400,0.999,0.484,0.435,0.639
Global_reactive_power,0.247,1.000,-0.112,0.266,0.123,0.139,0.090
Voltage,-0.400,-0.112,1.000,-0.411,-0.196,-0.167,-0.268
Global_intensity,0.999,0.266,-0.411,1.000,0.489,0.440,0.627
Sub_metering_1,0.484,0.123,-0.196,0.489,1.000,0.055,0.103
Sub_metering_2,0.435,0.139,-0.167,0.440,0.055,1.000,0.081
Sub_metering_3,0.639,0.090,-0.268,0.627,0.103,0.081,1.000


### Interpretación de las correlaciones

La matriz de correlación muestra una relación positiva extremadamente alta entre `Global_active_power` y `Global_intensity`, con un coeficiente de aproximadamente 0.999. Esto indica que ambas variables contienen información fuertemente relacionada y deberá considerarse posteriormente durante la selección de características. También se observan correlaciones positivas moderadas entre `Global_active_power` y los distintos submedidores, especialmente con `Sub_metering_3` (0.639). Este comportamiento es coherente con que los submedidores representan componentes específicos del consumo eléctrico total del hogar. Por otra parte, `Voltage` presenta correlaciones negativas moderadas con `Global_active_power` (-0.400) y `Global_intensity` (-0.411). Estas relaciones no se consideran errores de calidad y no justifican la eliminación automática de variables. Se documentan porque pueden generar redundancia o adquirir especial importancia dependiendo de la variable objetivo y de la estrategia de modelado utilizada posteriormente.

### Evaluación preliminar del riesgo de data leakage

El riesgo de data leakage no puede determinarse únicamente mediante las correlaciones observadas, ya que depende de la variable objetivo, del horizonte de predicción y de la información que estará disponible en el momento de realizar cada pronóstico.

Sin embargo, el diagnóstico permite identificar riesgos que deberán controlarse durante la construcción del dataset de modelado. En particular, `Global_active_power` y `Global_intensity` presentan una correlación de aproximadamente 0.999, por lo que deberá evaluarse cuidadosamente el uso de mediciones contemporáneas si alguna de estas variables se utiliza como objetivo. Asimismo, cualquier característica temporal derivada deberá construirse utilizando únicamente información disponible hasta el instante de predicción. Variables rezagadas, medias móviles y otras características basadas en la serie deberán evitar incorporar observaciones futuras.

Finalmente, debido a la naturaleza temporal del dataset, la separación entre entrenamiento, validación y prueba deberá respetar el orden cronológico de las observaciones y evitar particiones aleatorias que puedan introducir información futura en el entrenamiento. En esta etapa no se eliminarán variables por posible leakage. La decisión se realizará posteriormente cuando se definan formalmente la variable objetivo, el horizonte de predicción y el conjunto de características del modelo.

## 11. Conclusiones del diagnóstico de calidad

El diagnóstico permitió evaluar la calidad y estructura del dataset antes de realizar cualquier modificación sobre los datos originales.

Los principales hallazgos fueron los siguientes:

- **Valores faltantes:** se identificaron 25,979 registros afectados, equivalentes aproximadamente al 1.25% del dataset. En seis variables eléctricas los valores ausentes están representados mediante el símbolo `?`, mientras que en `Sub_metering_3` se reconocieron como valores nulos. Los valores faltantes ocurren simultáneamente en las siete mediciones eléctricas, mientras que `Date` y `Time` permanecen disponibles.

- **Duplicados:** no se encontraron registros completamente duplicados ni timestamps repetidos, por lo que cada combinación de fecha y hora representa una única observación.

- **Fechas y tipos de datos:** los 2,075,259 registros pudieron formar timestamps válidos. Asimismo, una vez excluidos los valores faltantes identificados, las siete variables eléctricas pueden convertirse correctamente a valores numéricos.

- **Continuidad temporal:** la serie presenta una frecuencia constante de un minuto. Todas las diferencias entre timestamps consecutivos fueron de exactamente un minuto, sin detectarse intervalos temporales irregulares.

- **Valores imposibles y outliers:** no se encontraron valores negativos en las variables eléctricas. El método IQR identificó valores estadísticamente atípicos en varias variables, pero el análisis posterior mostró que no existe evidencia suficiente para considerarlos errores de calidad. Por esta razón, no se plantea su eliminación automática.

- **Distribución de las variables:** varias mediciones presentan asimetría positiva. Esta característica es especialmente marcada en `Sub_metering_1` y `Sub_metering_2`, lo que ayuda a explicar el comportamiento observado durante la detección de outliers.

- **Correlaciones:** se identificó una correlación positiva extremadamente alta entre `Global_active_power` y `Global_intensity` (aproximadamente 0.999), además de relaciones moderadas entre el consumo global y algunos de los submedidores. Estas relaciones deberán considerarse posteriormente durante la selección de características.

- **Riesgo de data leakage:** no se eliminaron variables por leakage durante esta etapa, ya que su evaluación definitiva depende de la variable objetivo y del horizonte de predicción. El futuro proceso de modelado deberá respetar el orden temporal y garantizar que las características utilizadas contengan únicamente información disponible hasta el momento de cada predicción.

### Resultado del diagnóstico

El dataset presenta una estructura temporal consistente y no se detectaron problemas relacionados con duplicados, timestamps inválidos, frecuencia irregular o valores negativos. El principal problema de calidad identificado corresponde a los 25,979 registros con mediciones eléctricas faltantes.

A partir de estos hallazgos, la siguiente etapa se enfocará en aplicar y justificar las decisiones de limpieza, convertir las variables a sus tipos definitivos y validar que el dataset resultante conserve correctamente su estructura temporal.